In [1]:
import pdfplumber
import pandas as pd
import numpy as np

def extract_dengue_table(pdf_path):
    # Open the PDF file
    with pdfplumber.open(pdf_path) as pdf:
        # Page 8 is at index 7 (0-based index)
        page = pdf.pages[7]
        
        # Extract the table
        # 'vertical_strategy': 'lines' helps detect grid lines in these reports
        # 'intersection_tolerance': 15 helps with slightly misaligned lines
        table = page.extract_table({
            "vertical_strategy": "lines", 
            "horizontal_strategy": "lines",
            "intersection_tolerance": 15
        })

    if not table:
        print("No table found on Page 8.")
        return None

    # Create DataFrame
    # Note: The first few rows are usually nested headers. We skip them to set our own.
    # Adjust the slice [2:] if the header takes up more or fewer rows.
    df = pd.DataFrame(table[2:])

    # Define English Column Names based on the report structure 
    # The columns typically follow this order:
    # 1. Division | 2. Serial | 3. Hospital/District | 4. Govt (New) | 5. Pvt (New) 
    # 6. Total (New) | 7. Cumulative Total | 8. Death | 9. Discharged | 10. Currently Admitted
    english_columns = [
        "Division", 
        "Serial_No", 
        "Hospital_District_Name", 
        "New_Admitted_Govt", 
        "New_Admitted_Private", 
        "New_Admitted_Total", 
        "Cumulative_Admitted_Total", 
        "Cumulative_Death", 
        "Discharged_Patients", 
        "Currently_Admitted"
    ]

    # Assign columns (ensure length matches)
    if len(df.columns) == len(english_columns):
        df.columns = english_columns
    else:
        # Fallback if column count doesn't match perfectly (sometimes hidden columns exist)
        print(f"Warning: Extracted {len(df.columns)} columns, expected {len(english_columns)}")
        # You might need to adjust the column list if this hits
        df.columns = [f"Col_{i}" for i in range(len(df.columns))]

    # --- Data Cleaning ---

    # 1. Handle Merged Division Names
    # In the PDF, "Dhaka Division" is written once and merged down.
    # We replace empty strings/None with NaN and forward fill.
    df["Division"] = df["Division"].replace(["", None], np.nan)
    df["Division"] = df["Division"].ffill()

    # 2. Clean Numeric Columns
    # Convert numeric columns to numbers, coercing errors (non-numbers become NaN)
    numeric_cols = [
        "New_Admitted_Govt", "New_Admitted_Private", "New_Admitted_Total",
        "Cumulative_Admitted_Total", "Cumulative_Death", 
        "Discharged_Patients", "Currently_Admitted"
    ]
    
    for col in numeric_cols:
        # Remove potential commas or spaces before converting
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

    # 3. Remove "Noise" Rows
    # Filter out rows where the Hospital Name is empty or contains the header text repeated
    df = df.dropna(subset=["Hospital_District_Name"])
    
    # Optional: Filter out rows that are just sub-headers if they exist
    df = df[df["Hospital_District_Name"] != "Hospital_District_Name"]

    return df

# --- Usage ---
file_path = "20260126_dengue_all.pdf"  # Make sure this matches your file name
df_result = extract_dengue_table(file_path)

# Display the first few rows
print(df_result.head())

# Export to CSV (optional)
# df_result.to_csv("dengue_report_page_8.csv", index=False)

KeyError: 'Division'

In [2]:


def extract_dengue_table_robust(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        # Page 8 is at index 7
        page = pdf.pages[7]
        
        # 'vertical_strategy': 'lines' is best, but sometimes 'text' helps if lines are faint.
        # We stick to 'lines' but add snap_tolerance to catch slightly offset lines.
        table = page.extract_table({
            "vertical_strategy": "lines", 
            "horizontal_strategy": "lines",
            "snap_tolerance": 4,  # Helps catch lines that don't perfectly intersect
        })

    if not table:
        raise ValueError("No table found on Page 8. Check page index.")

    # 1. Inspect Raw Data (Crucial for debugging)
    print(f"DEBUG: Extracted {len(table[0])} columns.")
    
    # 2. Create DataFrame & Drop Header Rows
    # Usually rows 0-1 are headers. Adjust if data starts earlier/later.
    df = pd.DataFrame(table[2:]) 

    # 3. Define Expected Columns (Order matters!)
    # We define the core columns we care about from Left to Right.
    expected_headers = [
        "Division", 
        "Serial_No", 
        "Hospital_District_Name", 
        "New_Admitted_Govt", 
        "New_Admitted_Private", 
        "New_Admitted_Total", 
        "Cumulative_Admitted_Total", 
        "Death", 
        "Discharged_Patients", 
        "Currently_Admitted"
    ]

    # 4. ROBUST MAPPING: Map as many columns as exist
    # If we found MORE columns than expected, we keep the first 10 and label the rest "Extra".
    # If we found FEWER, we map what we can.
    
    current_col_count = len(df.columns)
    
    if current_col_count >= len(expected_headers):
        # We have enough columns (or too many). Map the first 10 strictly.
        # Any extra columns get generic names.
        new_columns = expected_headers + [f"Extra_{i}" for i in range(current_col_count - len(expected_headers))]
        df.columns = new_columns
    else:
        # We have MISSING columns. This is dangerous, but we map left-to-right.
        print(f"WARNING: Found only {current_col_count} columns. Some data might be merged.")
        df.columns = expected_headers[:current_col_count]

    # 5. Check if 'Division' exists before processing
    if "Division" not in df.columns:
        raise KeyError(f"Could not map 'Division' column. Columns found: {df.columns.tolist()}")

    # --- Data Cleaning ---

    # Handle Merged Division Names (Forward Fill)
    df["Division"] = df["Division"].replace(["", None], np.nan)
    df["Division"] = df["Division"].ffill()

    # Clean Numeric Columns (Remove commas, convert to numeric)
    # We iterate only through columns that actually exist in the df
    numeric_targets = [
        "New_Admitted_Govt", "New_Admitted_Private", "New_Admitted_Total",
        "Cumulative_Admitted_Total", "Death", 
        "Discharged_Patients", "Currently_Admitted"
    ]
    
    for col in numeric_targets:
        if col in df.columns:
            # Convert to string -> remove commas -> to numeric
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '').replace('None', ''), 
                errors='coerce'
            )

    # Filter out empty rows or sub-header repetitions
    if "Hospital_District_Name" in df.columns:
        df = df.dropna(subset=["Hospital_District_Name"])
        # Remove rows that might be accidental header captures
        df = df[df["Hospital_District_Name"] != "Hospital_District_Name"]

    return df

# --- Execution ---
file_path = "20260126_dengue_all.pdf"
try:
    df_result = extract_dengue_table_robust(file_path)
    print("Success! Table Extracted.")
    print(df_result.head())
    print(f"\nDataFrame Shape: {df_result.shape}")
except Exception as e:
    print(e)

DEBUG: Extracted 11 columns.
Success! Table Extracted.
     Division   Serial_No Hospital_District_Name  New_Admitted_Govt  \
0       রেভাগ  ক্রর ক নিং                   বজলা                NaN   
3  ঢাকা হানগি           ১                   ঢাকা                NaN   
4  ঢাকা হানগি           ২                ফরিদপুি                NaN   
5  ঢাকা হানগি           ৩                ফরিদপুি                NaN   
7  ঢাকা হানগি           ৪                গাজীপুি                NaN   

   New_Admitted_Private  New_Admitted_Total  Cumulative_Admitted_Total  Death  \
0                   NaN                 NaN                        NaN    NaN   
3                   NaN                 NaN                        NaN    NaN   
4                   NaN                 NaN                        NaN    NaN   
5                   NaN                 NaN                        NaN    NaN   
7                   NaN                 NaN                        NaN    NaN   

   Discharged_Patients  Current

In [4]:
df_result.head()

,Division,Serial_No,Hospital_District_Name,New_Admitted_Govt,New_Admitted_Private,New_Admitted_Total,Cumulative_Admitted_Total,Death,Discharged_Patients,Currently_Admitted,Extra_0
0,রেভাগ,ক্রর ক নিং,বজলা,NaN,NaN,NaN,NaN,NaN,NaN,NaN,েত ব ামন ভরত ববিাগী
3,ঢাকা হানগি,১,ঢাকা,NaN,NaN,NaN,NaN,NaN,NaN,NaN,৩
4,ঢাকা হানগি,২,ফরিদপুি,NaN,NaN,NaN,NaN,NaN,NaN,NaN,০
5,ঢাকা হানগি,৩,ফরিদপুি,NaN,NaN,NaN,NaN,NaN,NaN,NaN,০
7,ঢাকা হানগি,৪,গাজীপুি,NaN,NaN,NaN,NaN,NaN,NaN,NaN,৫


In [5]:

def extract_dengue_data_robust(pdf_path):
    # 1. Define Translation Dictionaries
    # Map Bangla numerals to English
    bangla_to_eng_digits = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

    # Map all 64 Districts of Bangladesh (Bangla to English)
    # Note: Includes common spelling variations found in DGHS reports
    district_map = {
        # Dhaka Division
        'ঢাকা': 'Dhaka', 'ফরিদপুর': 'Faridpur', 'গাজীপুর': 'Gazipur', 'গোপালগঞ্জ': 'Gopalganj',
        'কিশোরগঞ্জ': 'Kishoreganj', 'মাদারীপুর': 'Madaripur', 'মানিকগঞ্জ': 'Manikganj',
        'মুন্সিগঞ্জ': 'Munshiganj', 'নারায়ণগঞ্জ': 'Narayanganj', 'নরসিংদী': 'Narsingdi',
        'রাজবাড়ী': 'Rajbari', 'শরীয়তপুর': 'Shariatpur', 'টাঙ্গাইল': 'Tangail',
        
        # Chattogram Division
        'চট্টগ্রাম': 'Chattogram', 'ব্রাহ্মণবাড়িয়া': 'Brahmanbaria', 'ব্রাহ্মনবাড়িয়া': 'Brahmanbaria',
        'কুমিল্লা': 'Cumilla', 'কক্সবাজার': 'Cox\'s Bazar', 'চাঁদপুর': 'Chandpur',
        'ফেনী': 'Feni', 'খাগড়াছড়ি': 'Khagrachhari', 'লক্ষ্মীপুর': 'Lakshmipur', 'লক্ষীপুর': 'Lakshmipur',
        'নোয়াখালী': 'Noakhali', 'রাঙ্গামাটি': 'Rangamati', 'বান্দরবান': 'Bandarban',
        
        # Rajshahi Division
        'রাজশাহী': 'Rajshahi', 'বগুড়া': 'Bogura', 'চাপাইনবাবগঞ্জ': 'Chapainawabganj',
        'জয়পুরহাট': 'Joypurhat', 'নওগাঁ': 'Naogaon', 'নাটোর': 'Natore',
        'পাবনা': 'Pabna', 'সিরাজগঞ্জ': 'Sirajganj',
        
        # Khulna Division
        'খুলনা': 'Khulna', 'বাগেরহাট': 'Bagerhat', 'চুয়াডাঙ্গা': 'Chuadanga',
        'যশোর': 'Jashore', 'ঝিনাইদহ': 'Jhenaidah', 'কুষ্টিয়া': 'Kushtia',
        'মাগুরা': 'Magura', 'মেহেরপুর': 'Meherpur', 'নড়াইল': 'Narail', 'সাতক্ষীরা': 'Satkhira',
        
        # Barishal Division
        'বরিশাল': 'Barishal', 'বরগুনা': 'Barguna', 'ভোলা': 'Bhola',
        'ঝালকাঠি': 'Jhalokathi', 'পটুয়াখালী': 'Patuakhali', 'পিরোজপুর': 'Pirojpur',
        
        # Sylhet Division
        'সিলেট': 'Sylhet', 'হবিগঞ্জ': 'Habiganj', 'মৌলভীবাজার': 'Moulvibazar', 'সুনামগঞ্জ': 'Sunamganj',
        
        # Rangpur Division
        'রংপুর': 'Rangpur', 'দিনাজপুর': 'Dinajpur', 'গাইবান্ধা': 'Gaibandha',
        'কুড়িগ্রাম': 'Kurigram', 'লালমনিরহাট': 'Lalmonirhat', 'নীলফামারী': 'Nilphamari',
        'পঞ্চগড়': 'Panchagarh', 'ঠাকুরগাঁও': 'Thakurgaon',
        
        # Mymensingh Division
        'ময়মনসিংহ': 'Mymensingh', 'জামালপুর': 'Jamalpur', 'নেত্রকোণা': 'Netrokona', 
        'নেত্রকোনা': 'Netrokona', 'শেরপুর': 'Sherpur'
    }

    # 2. Extract Table
    with pdfplumber.open(pdf_path) as pdf:
        # Page 8 is index 7
        page = pdf.pages[7]
        table = page.extract_table({
            "vertical_strategy": "lines", 
            "horizontal_strategy": "lines",
            "snap_tolerance": 4
        })

    if not table:
        return None

    # 3. Create DataFrame & Drop Header Rows
    # Skipping first 2 rows usually removes the messy headers
    df = pd.DataFrame(table[2:])
    
    # Manually assign columns based on visual inspection of the report structure
    # Typically: Division | Serial | District/Hospital | Govt | Pvt | TOTAL (24h) | ...
    # We map enough columns to capture what we need
    required_cols = [
        "Division", "Serial", "Raw_District_Name", 
        "Govt_Cases", "Pvt_Cases", "Total_Cases_24h"
    ]
    
    # Handle column mismatch safely
    current_cols = len(df.columns)
    if current_cols >= len(required_cols):
        df.columns = required_cols + [f"Col_{i}" for i in range(current_cols - len(required_cols))]
    else:
        df.columns = required_cols[:current_cols]

    # 4. DATA CLEANING PIPELINE
    
    # A. Clean "Total_Cases_24h" (The Critical Fix)
    def clean_bangla_numerals(text):
        if not text: return 0
        text = str(text).strip()
        # Translate Bangla digits to English
        text = text.translate(bangla_to_eng_digits)
        # Remove commas or other noise
        text = text.replace(',', '').replace('-', '0') 
        try:
            return int(text)
        except ValueError:
            return 0

    df['Total_Cases_24h'] = df['Total_Cases_24h'].apply(clean_bangla_numerals)

    # B. Extract & Translate District Name
    # The column usually contains mixed data (Hospitals AND Districts).
    # We try to map the value. If it's a known district, we keep it.
    
    def map_district(text):
        if not text: return None
        # Clean text: remove newlines, spaces
        clean_text = str(text).strip().replace('\n', ' ')
        
        # Check against dictionary
        # We look for the district name inside the string (e.g., "Faridpur Medical" -> Faridpur)
        for bn_name, en_name in district_map.items():
            if bn_name in clean_text:
                return en_name
        return "Other/Hospital" # Label for rows that aren't clearly districts

    df['District_English'] = df['Raw_District_Name'].apply(map_district)

    # 5. Filter & Aggregate
    # Filter only rows that were identified as Districts (or group by them)
    # The user asked for "District, Total number of cases".
    # We group by the English District name and Sum the cases to be safe.
    
    final_df = df.groupby('District_English', as_index=False)['Total_Cases_24h'].sum()
    
    # Remove the "Other/Hospital" category if you strictly want districts
    final_df = final_df[final_df['District_English'] != "Other/Hospital"]

    return final_df

# --- Execution ---
file_path = "20260126_dengue_all.pdf"
try:
    df_clean = extract_dengue_data_robust(file_path)
    
    print("Successfully Extracted Data:")
    print(df_clean.head(10)) # Show first 10
    
    # Check for specific known districts to verify
    print("\nVerification (Specific Districts):")
    print(df_clean[df_clean['District_English'].isin(['Dhaka', 'Faridpur', 'Gazipur'])])

except Exception as e:
    print(f"Error: {e}")

Successfully Extracted Data:
  District_English  Total_Cases_24h
0            Dhaka                0
2          Tangail                0

Verification (Specific Districts):
  District_English  Total_Cases_24h
0            Dhaka                0


In [6]:
df_clean.head(10)

,District_English,Total_Cases_24h
0,Dhaka,0
2,Tangail,0


In [9]:
import pdfplumber
import pandas as pd
import re

# --- Configuration ---
# Page 8 dimensions are typically ~595 points wide (A4).
# We define "Zones" for the columns we need.
# You might need to adjust 'min_x' and 'max_x' after the first run if alignment is off.
COLUMN_ZONES = {
    "District_Garbage": {"min_x": 80, "max_x": 200},  # Left side: District Names
    "Total_Cases_24h":  {"min_x": 280, "max_x": 350}  # Middle: The 'Total' column
}

def clean_bangla_num(text):
    """Robust Bangla numeral cleaner."""
    if not text: return 0
    # Map specifically observed garbage numerals if necessary, plus standard Bangla
    trans = str.maketrans("০১২৩৪৫৬৭৮৯-", "01234567890")
    clean = text.translate(trans)
    # Extract first valid integer found
    match = re.search(r'\d+', clean)
    return int(match.group()) if match else 0

def extract_spatial_dengue(pdf_path):
    data = []
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[7] # Page 8
        words = page.extract_words(x_tolerance=3, y_tolerance=3)
        
        # 1. Group words by Row (using 'top' coordinate)
        # We round 'top' to nearest 5 points to group words on the same line
        rows = {}
        for w in words:
            row_y = round(w['top'] / 5) * 5
            if row_y not in rows: rows[row_y] = []
            rows[row_y].append(w)
            
        # 2. Process each row
        sorted_y = sorted(rows.keys())
        
        for y in sorted_y:
            row_words = rows[y]
            
            # Bucket text into our defined zones
            district_text = []
            cases_text = []
            
            for w in row_words:
                x = w['x0']
                text = w['text']
                
                # Check which zone this word falls into
                if COLUMN_ZONES["District_Garbage"]["min_x"] <= x <= COLUMN_ZONES["District_Garbage"]["max_x"]:
                    district_text.append(text)
                elif COLUMN_ZONES["Total_Cases_24h"]["min_x"] <= x <= COLUMN_ZONES["Total_Cases_24h"]["max_x"]:
                    cases_text.append(text)
            
            # 3. Compile Row Data
            if district_text: # Only keep rows that have a district name
                full_dist_str = " ".join(district_text)
                full_case_str = " ".join(cases_text)
                
                # Filter out header rows (checking if 'cases' is a number)
                cleaned_cases = clean_bangla_num(full_case_str)
                
                # If we found a district-like row, save it
                # (You can add logic here to skip the header row if it parses to 0)
                data.append({
                    "Raw_Garbage_Name": full_dist_str,
                    "Total_Cases_24h": cleaned_cases,
                    "Y_Pos": y # Debugging aid
                })

    df = pd.DataFrame(data)
    
    # --- Post-Processing: Map the Garbage ---
    # Since the font is broken, we map the *specific garbage strings* you are seeing.
    # Update this dictionary based on the 'Raw_Garbage_Name' output you see.
    garbage_map = {
        'ঢাকা হানগি': 'Dhaka',
        'ফরিদপি': 'Faridpur',
        'গাজীপি': 'Gazipur',
        'বগাপালগঞ্জ': 'Gopalganj',
        'রকমশিাগঞ্জ': 'Kishoreganj',
        '০ারকনগঞ্জ': 'Narayanganj', # Likely guess
        'মুরিরগঞ্জ': 'Munshiganj',  # Likely guess
        'নিরসিংদী': 'Narsingdi',    # Likely guess
        'টাঙ্গাইল': 'Tangail'       # Often survives intact
    }
    
    # Apply map, keep original if not found
    df['District_English'] = df['Raw_Garbage_Name'].map(garbage_map).fillna(df['Raw_Garbage_Name'])
    
    return df

# Execution
df_spatial = extract_spatial_dengue("20260126_dengue_all.pdf")
df_spatial.head(15)

,Raw_Garbage_Name,Total_Cases_24h,Y_Pos,District_English
0,স্বাস্থ্ু অরিদপ্তমিি,0,155,স্বাস্থ্ু অরিদপ্তমিি
1,রেভাগ ক্রর ক নিং,0,175,রেভাগ ক্রর ক নিং
2,১,0,205,১
3,২,0,210,২
4,৩,0,225,৩
5,৪,0,250,৪
6,৫,0,265,৫
7,৬,0,280,৬
8,৭,0,290,৭
9,৮,0,305,৮
